# Phase 2 — Fine-tune Gemma-4-E4B-it (LoRA / QLoRA) trên VLSP DateArith VI, sau đó đánh giá lại

Notebook này mirror **chính xác** flow của `run_finetune_phase2_colab.ipynb` (Qwen3-8B)  
nhưng thay bằng **`google/gemma-4-E4B-it`** với các điểm khác biệt sau:

| | Qwen3-8B | Gemma-4-E4B-it |
|---|---|---|
| Precision | bf16 (A100) | bf16 (A100) / fp16+4bit (T4) — auto-detect |
| System prompt | `<\|im_start\|>system` | Merge vào first user turn |
| Loss mask | `<\|im_start\|>assistant\n` | `<start_of_turn>model\n` |
| Thinking mode | `enable_thinking=True/False` | Không hỗ trợ — luôn `False` |
| Wrapper | `QwenChatLM` | `GemmaChatLM` |

**Flow:**
1. **TRAIN**: SFT LoRA (r=16) trên rows 1500–2999 của `date_training_dataset.txt` (1350 train / 150 val).
2. **EVAL**: zero_shot + few_shot trên `vlsp_date` (in-domain) và `vlsp_duration` (cross-task) với adapter.

⚠️ **A100**: `bf16=True`, `load_in_4bit=False` (default). **T4**: auto-switch sang QLoRA (`load_in_4bit=True`, `fp16=True`).

## Setup

In [ ]:
# === SETUP 1 — cài môi trường (thêm peft / datasets cho training) ===
# Colab pre-install torchao (cũ) xung đột với peft mới → uninstall trước.
!pip uninstall -q -y torchao
# sympy mặc định của Colab thiếu attribute 'printing' → gây lỗi cho transformers.
!pip install -q -U sympy
!pip install -q -U transformers accelerate scikit-learn pyyaml sentence-transformers faiss-cpu
!pip install -q -U peft datasets
!pip install -q bitsandbytes  # QLoRA cho T4 / P100
# ⚠️ Sau khi chạy cell này: Runtime → Restart session rồi chạy tiếp từ SETUP 2.

In [ ]:
# === SETUP 2 — tùy chọn mount Drive + clone repo ===
import os

USE_DRIVE = False  # True: mount Drive để persist Dataset / output
REPO_URL  = 'https://github.com/<YOUR_USER>/Temporal_Reasoning.git'  # TODO: đổi
REPO_DIR  = '/content/Temporal_Reasoning'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
print('CWD:', os.getcwd())
print('USE_DRIVE:', USE_DRIVE)

In [ ]:
# === SETUP 3 — paths + GPU detect ===
import os
import sys
import torch

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, REPO_DIR)

if USE_DRIVE:
    DRIVE_OUT_FT = '/content/drive/MyDrive/Temporal_Reasoning/outputs_ft_gemma4_e4b'
    DATASET_ROOT = '/content/drive/MyDrive/Temporal_Reasoning/Dataset'
    os.makedirs(DRIVE_OUT_FT, exist_ok=True)
    local_ds = os.path.join(REPO_DIR, 'Dataset')
    if not os.path.exists(local_ds):
        os.symlink(DATASET_ROOT, local_ds)
    print('Dataset ->', os.readlink(local_ds) if os.path.islink(local_ds) else local_ds)
else:
    DRIVE_OUT_FT = '/content/outputs_ft_gemma4_e4b'
    DATASET_ROOT = os.path.join(REPO_DIR, 'Dataset')
    os.makedirs(DRIVE_OUT_FT, exist_ok=True)
    if not os.path.exists(DATASET_ROOT):
        raise FileNotFoundError(f'Không tìm thấy Dataset tại {DATASET_ROOT}.')
    print('Dataset ->', DATASET_ROOT)

# Checkpoint local — mất khi Colab disconnect (đúng lựa chọn user).
CHECKPOINT_DIR = '/content/checkpoints/gemma4_e4b_lora_vlsp_date'
os.makedirs(os.path.dirname(CHECKPOINT_DIR), exist_ok=True)

# ── GPU detect: auto-switch QLoRA cho T4 / P100 (VRAM < 20 GB) ───────────────
if torch.cuda.is_available():
    _gpu   = torch.cuda.get_device_name(0)
    _vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_QLORA = _vram < 20.0
    print(f'GPU: {_gpu}  |  VRAM: {_vram:.1f} GB  |  Mode: {"QLoRA" if USE_QLORA else "LoRA bf16"}')
else:
    USE_QLORA = False
    print('No GPU — training will be very slow')

print('DRIVE_OUT_FT :', DRIVE_OUT_FT)
print('CHECKPOINT   :', CHECKPOINT_DIR)

In [ ]:
# === SETUP 4 — preprocess raw → JSONL (Dataset/Preprocessed/) ===
!python -m src.data.preprocess

## TRAIN — SFT LoRA / QLoRA

Thời gian dự kiến:
- **A100 40 GB** (LoRA bf16): ~8–12 phút (3 epoch × ~169 steps)
- **T4 16 GB** (QLoRA 4-bit): ~20–30 phút

Smoke test nhanh: set `raw['num_epochs'] = 1; raw['train_pool_size'] = 50` trước khi `train_sft`.

In [ ]:
# === TRAIN — chạy SFT LoRA, save adapter vào CHECKPOINT_DIR ===
import yaml
from src.training.sft import SFTRunConfig, train_sft

with open('configs/sft_gemma4_e4b_lora.yaml', encoding='utf-8') as f:
    raw = yaml.safe_load(f)

# Override output path → local Colab
raw['output_dir'] = CHECKPOINT_DIR

# ── Gemma-specific: patch precision for T4 (QLoRA) vs A100 (LoRA bf16) ────────
# Qwen version dùng bf16 cố định; Gemma cần auto-switch vì T4 không hỗ trợ bf16.
if USE_QLORA:
    raw['load_in_4bit'] = True
    raw['bf16']         = False
    raw['fp16']         = True
    print('[TRAIN] T4 detected → QLoRA (4-bit, fp16)')
else:
    raw['load_in_4bit'] = False
    raw['bf16']         = True
    raw['fp16']         = False
    print('[TRAIN] A100 detected → LoRA bf16')

cfg = SFTRunConfig(**raw)
print('SFT config:', cfg)

ADAPTER_PATH = train_sft(cfg)
print('Adapter saved to:', ADAPTER_PATH)
!ls -lah $ADAPTER_PATH

## EVAL — chạy lại zero_shot + few_shot × 2 dataset với adapter

Load model (base + adapter) **một lần** để reuse cho 4 run — tiết kiệm thời gian.  
Output FT ghi vào `outputs_ft_gemma4_e4b/`, summary.csv riêng → dễ so sánh baseline vs FT.

⚠️ Khác Qwen: `GemmaChatLM` thay `QwenChatLM`; `enable_thinking` luôn `False`.

In [ ]:
# === EVAL setup — load Gemma-4-E4B + adapter 1 lần ===
# Điểm khác biệt so với Qwen: GemmaChatLM thay QwenChatLM;
# load_in_4bit theo USE_QLORA; enable_thinking luôn False.
from src.models.gemma import GemmaChatLM, GemmaConfig
from src.runner import load_config, run

MODEL_FT = GemmaChatLM(GemmaConfig(
    model_name='google/gemma-4-E4B',
    dtype='bfloat16' if not USE_QLORA else 'float16',
    load_in_4bit=USE_QLORA,
    adapter_path=ADAPTER_PATH,
))
MODEL_FT.load()
print('Model + adapter ready:', MODEL_FT.config.model_name, '+', MODEL_FT.config.adapter_path)


def run_exp_ft(cfg_path, *, verbose=True, verbose_first_n=5, verbose_every=200,
               running_score_every=100, output_dir=DRIVE_OUT_FT):
    """Mirror run_exp_ft() từ Qwen notebook:
    - suffix '_ft' vào experiment_name
    - trỏ output về outputs_ft_gemma4_e4b/
    - chạy với MODEL_FT (Gemma + adapter)
    - luôn enable_thinking=False (Gemma không hỗ trợ)
    """
    cfg = load_config(cfg_path)
    cfg.output_dir          = output_dir
    cfg.experiment_name     = cfg.experiment_name + '_ft'
    cfg.model_name          = 'google/gemma-4-E4B'
    cfg.enable_thinking     = False   # Gemma không có thinking mode
    cfg.adapter_path        = ADAPTER_PATH  # log vào metrics.json config
    cfg.verbose             = verbose
    cfg.verbose_first_n     = verbose_first_n
    cfg.verbose_every       = verbose_every
    cfg.running_score_every = running_score_every
    return run(cfg, model=MODEL_FT)

### `vlsp_date` (in-domain) — kỳ vọng accuracy tăng rõ rệt so với baseline.

In [ ]:
# === EXP 1/4 — zero_shot × vlsp_date (FT) ===
m = run_exp_ft('configs/zero_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 2/4 — few_shot k=3 × vlsp_date (FT) ===
m = run_exp_ft('configs/few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

### `vlsp_duration` (cross-task, cùng VI) — kiểm tra catastrophic forgetting.

In [ ]:
# === EXP 3/4 — zero_shot × vlsp_duration (FT) ===
m = run_exp_ft('configs/zero_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 4/4 — few_shot k=4 × vlsp_duration (FT) ===
m = run_exp_ft('configs/few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

## So sánh baseline vs FT

In [ ]:
# === COMPARE — đọc từ metrics.json (robust với ragged CSV) ===
# Mirror chính xác logic từ Qwen notebook — chỉ đổi BASE_DIR.
import glob
import json
import os
import pandas as pd

BASE_DIR = '/content/outputs' if not USE_DRIVE \
    else '/content/drive/MyDrive/Temporal_Reasoning/outputs'
FT_DIR   = DRIVE_OUT_FT

def _scan_metrics(root: str, label: str):
    """Glob outputs/<method>/<dataset>/metrics.json -> DataFrame."""
    rows = []
    for path in glob.glob(os.path.join(root, '*', '*', 'metrics.json')):
        with open(path, encoding='utf-8') as f:
            d = json.load(f)
        m = d.get('metrics', {})
        if 'f1' in m:
            metric, score = 'f1', m['f1']
        elif 'accuracy' in m:
            metric, score = 'accuracy', m['accuracy']
        else:
            metric, score = '', float('nan')
        rows.append({
            'method':              d.get('method'),
            'dataset':             d.get('dataset'),
            'k_shot':              d.get('k_shot'),
            'metric':              metric,
            f'score_{label}':      round(score, 4) if score == score else None,
            f'parse_fail_{label}': m.get('parse_fail', 0),
        })
    if not rows:
        print(f'[warn] không tìm thấy metrics.json nào dưới {root}')
        return None
    return pd.DataFrame(rows)

base = _scan_metrics(BASE_DIR, 'base')
ft   = _scan_metrics(FT_DIR,   'ft')

if base is not None and ft is not None:
    cmp = base.merge(ft, on=['method', 'dataset', 'k_shot', 'metric'], how='outer')
    cmp['delta'] = cmp['score_ft'] - cmp['score_base']
    print(cmp.sort_values(['dataset', 'method']).to_string(index=False))
else:
    print('Skip compare — thiếu metrics.json.')

In [ ]:
# === EXPORT — zip outputs_ft để tải về trước khi Colab disconnect ===
import os
import shutil
from google.colab import files

os.chdir('/content')
out_dir     = os.path.abspath(DRIVE_OUT_FT)
parent_dir  = os.path.dirname(out_dir)
folder_name = os.path.basename(out_dir.rstrip('/'))
zip_path    = shutil.make_archive(
    f'/content/{folder_name}', 'zip',
    root_dir=parent_dir, base_dir=folder_name,
)
print('Created zip:', zip_path)
files.download(zip_path)